# 05 E5: shortcuts, mitigation (and later calibration)

All models are the final ones (chosen on dev, notebooks 02-03), refit from code with seed 0.
Test-time changes (`deepvoice.features.apply_transform`): **trim** (cut leading/trailing silence
with the group-S VAD), **gain-12** (12 dB quieter), **gain+6** (6 dB louder, clipped). Features of
changed audio are cached as `<group>_<split>_<change>.parquet`.

- **H1b** (frozen): trimming silence raises the rung 1 eval EER by >= 5 pp with the paired
  speaker-bootstrap CI excluding 0.
- XLS-R normalizes every clip to zero mean and unit variance, so gain changes cannot change its
  input (apart from clipped samples at +6 dB); it gets the trim test only.
- **Mitigation (E5.2):** rung 1 retrained on augmented train audio (trim, random 0-1 s noise padding,
  random -12..+6 dB gain; seeded per clip), same hyperparameters, compared on the same clips.

Results: `results/e5.csv` (standard columns; `split` names the change) and
`results/e5_diffs.csv` (paired differences with 95% CIs).

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
from deepvoice.data import load_itw
from deepvoice.features import feature_tables, load_xy
from deepvoice.metrics import paired_speaker_bootstrap_diff
from deepvoice.models import run

# cached features of changed audio (computed once; loaded here)
p = pd.read_parquet("data/protocols.parquet")
sets = {"eval": p[p.split == "eval"].reset_index(drop=True), "itw": load_itw().reset_index(drop=True)}
for change in ["trim", "gain-12", "gain+6"]:
    for s, df in sets.items():
        feature_tables(df, f"{s}_{change}", transform=change)
feature_tables(p[p.split == "train"].reset_index(drop=True), "train_augment", transform="augment")

FS = {"groups": ("lfcc",), "n_coef": 40, "deltas": 2, "stats": ("mean", "std")}
gap = pd.read_csv(ROOT / "results/e3_gap.csv")
HGB = json.loads(gap[(gap.model == "hgb") & (gap["selected by"] == "dev")].params.iloc[0])
EV = {"dev": load_xy("dev", **FS)}
for s in ["eval", "itw"]:
    EV[s] = load_xy(s, **FS)
    for change in ["trim", "gain-12", "gain+6"]:
        EV[f"{s}_{change}"] = load_xy(s, variant=change, **FS)
train, train_aug = load_xy("train", **FS), load_xy("train", variant="augment", **FS)
X = list(train.columns[4:])
DATASET = lambda s: "itw" if s.startswith("itw") else "asvspoof"
print({k: len(v) for k, v in EV.items()})

## Rung 1: original vs augmented training, on clean and changed audio

In [ ]:
models, scores = {}, {}
for tag, tr in [("lfcc40dd_e3devsel", train), ("lfcc40dd_e3devsel_augment", train_aug)]:
    m, _ = run(1, "hgb", X, tr, [(DATASET(s), s, EV[s]) for s in EV], params=HGB,
               experiment="e5", feature_set=tag)
    models[tag] = m
    scores[tag] = {s: m.predict_proba(EV[s][X].to_numpy())[:, 1] for s in EV}
e5 = pd.read_csv(ROOT / "results/e5.csv")
e5[e5.rung == 1].assign(eer=lambda d: (100 * d.eer).round(2)).pivot_table(index="split", columns="model", values="eer")

## Paired differences (EER of B minus EER of A on the same clips, 95% CI)

In [ ]:
rows = []
orig, aug = scores["lfcc40dd_e3devsel"], scores["lfcc40dd_e3devsel_augment"]
for s in ["eval", "itw"]:
    for change in ["trim", "gain-12", "gain+6"]:  # same clips and labels, changed audio
        d, lo, hi = paired_speaker_bootstrap_diff(EV[s].label, orig[s], orig[f"{s}_{change}"], EV[s].speaker)
        rows.append({"model": "rung 1 (original)", "A": f"{s} clean", "B": f"{s} {change}", "clips": s,
                     "diff_pp": 100 * d, "ci_low_pp": 100 * lo, "ci_high_pp": 100 * hi})
    for v in ["", "_trim"]:
        d, lo, hi = paired_speaker_bootstrap_diff(EV[s].label, orig[s + v], aug[s + v], EV[s].speaker)
        rows.append({"model": "augmented minus original", "A": f"original on {s}{v.replace('_', ' ')}",
                     "B": f"augmented on {s}{v.replace('_', ' ')}", "clips": s,
                     "diff_pp": 100 * d, "ci_low_pp": 100 * lo, "ci_high_pp": 100 * hi})
diffs = pd.DataFrame(rows)
diffs.to_csv(ROOT / "results/e5_diffs.csv", index=False, float_format="%.3f")
diffs.round(2)

## H1b verdict (rule from the frozen `hypotheses.md`)

In [ ]:
h = diffs[(diffs.model == "rung 1 (original)") & (diffs.B == "eval trim")].iloc[0]
h1b = h.diff_pp >= 5 and h.ci_low_pp > 0
print(f"Trimming raises rung 1 eval EER by {h.diff_pp:.1f} pp [{h.ci_low_pp:.1f}, {h.ci_high_pp:.1f}] -> H1b {'TRUE' if h1b else 'FALSE'}")

## Rung 2 (frozen XLS-R layer 3 + logistic regression): clean vs trimmed

In [ ]:
from deepvoice.xlsr import LAYERS, extract_cached

extract_cached(sets["eval"], "eval_trim", transform="trim")   # cached: loads, never recomputes
extract_cached(sets["itw"], "itw_trim", transform="trim")
K, COLS = LAYERS.index(3), [f"x{i:04d}" for i in range(1024)]
META = {"train": train, "dev": EV["dev"], "eval": EV["eval"], "eval_trim": EV["eval"], "itw": EV["itw"], "itw_trim": EV["itw"]}


def xframe(name):
    meta = META[name]
    assert Path(f"data/features/xlsr300m_{name}_files.txt").read_text().split() == list(meta.file)
    e = np.asarray(np.load(f"data/features/xlsr300m_{name}.npy", mmap_mode="r")[:, K], dtype=np.float32)
    return pd.concat([meta[["file", "label", "speaker"]], pd.DataFrame(e, columns=COLS)], axis=1)


R2 = {s: xframe(s) for s in META}
m2, _ = run(2, "logreg", COLS, R2["train"], [(DATASET(s), s, R2[s]) for s in META if s != "train"],
            params={"C": 1.0}, experiment="e5", feature_set="xlsr300m_L03")
s2 = {s: m2.predict_proba(R2[s][COLS].to_numpy())[:, 1] for s in META if s != "train"}
for s in ["eval", "itw"]:
    d, lo, hi = paired_speaker_bootstrap_diff(EV[s].label, s2[s], s2[f"{s}_trim"], EV[s].speaker)
    rows.append({"model": "rung 2 (XLS-R)", "A": f"{s} clean", "B": f"{s} trim", "clips": s,
                 "diff_pp": 100 * d, "ci_low_pp": 100 * lo, "ci_high_pp": 100 * hi})
diffs = pd.DataFrame(rows)
diffs.to_csv(ROOT / "results/e5_diffs.csv", index=False, float_format="%.3f")
diffs[diffs.model == "rung 2 (XLS-R)"].round(2)

## Calibration (E5.3): reliability, ECE (15 bins), temperature scaling fit on dev

All three final models were trained with balanced class weights, so their probabilities assume a
50/50 world while the data is about 90% fake (ITW: 37%); temperature scaling (one number, no bias)
can soften over-confidence but cannot fix that prior shift.

In [ ]:
from deepvoice.metrics import apply_temperature, ece, fit_temperature, reliability
from deepvoice.models import build

SIL = ["lead_sil", "trail_sil", "sil_ratio", "duration"]
silf = {s: load_xy(s, groups=("silence",)) for s in ["train", "dev", "eval", "itw"]}
m0 = build("tree", 0, max_depth=3).fit(silf["train"][SIL].to_numpy(), silf["train"].label.to_numpy())
PROBS = {0: {s: m0.predict_proba(silf[s][SIL].to_numpy())[:, 1] for s in ["dev", "eval", "itw"]},
         1: {s: orig[s] for s in ["dev", "eval", "itw"]},
         2: {s: s2[s] for s in ["dev", "eval", "itw"]}}
cal = []
for rung, pr in PROBS.items():
    t = fit_temperature(EV["dev"].label, pr["dev"])
    for s in ["dev", "eval", "itw"]:
        y = EV[s].label
        cal.append({"rung": rung, "split": s, "temperature": t, "ece_before": ece(y, pr[s]),
                    "ece_after": ece(y, apply_temperature(pr[s], t))})
cal = pd.DataFrame(cal)
cal.to_csv(ROOT / "results/e5_calibration.csv", index=False, float_format="%.4f")
cal.assign(ece_before=lambda d: (100 * d.ece_before).round(1), ece_after=lambda d: (100 * d.ece_after).round(1))

## Failure analysis (E5.5): where do rungs 1 and 2 fail?

In [ ]:
from deepvoice.data import load_gender
from deepvoice.metrics import eer

gender = load_gender()
fail = []
def bucket_eers(rung, s, sc, factor, buckets):
    d = EV[s][["label"]].assign(score=sc, b=buckets)
    for b, g in d.groupby("b", observed=True):
        if g.label.nunique() == 2:
            fail.append({"rung": rung, "dataset": s, "factor": factor, "bucket": str(b),
                         "n": len(g), "eer": eer(g.label, g.score)[0]})

for rung, sc in [(1, orig), (2, s2)]:
    for s in ["eval", "itw"]:
        sf = silf[s]
        bucket_eers(rung, s, sc[s], "duration (s)", pd.cut(sf.duration, [0, 2, 3, 4, 6, 100]))
        bucket_eers(rung, s, sc[s], "silence ratio", pd.cut(sf.sil_ratio, [-0.01, 0.2, 0.35, 0.5, 1]))
    bucket_eers(rung, "eval", sc["eval"], "gender", EV["eval"].speaker.map(gender).fillna("unknown"))
fail = pd.DataFrame(fail)
fail.to_csv(ROOT / "results/e5_failures.csv", index=False, float_format="%.4f")
fail.assign(eer=lambda d: (100 * d.eer).round(1)).pivot_table(index=["dataset", "factor", "bucket"], columns="rung", values="eer")

In [ ]:
# per In-the-Wild speaker (rung 2): EER for each of the 54 speakers
spk = []
for name, g in EV["itw"].assign(score=s2["itw"]).groupby("speaker"):
    spk.append({"speaker": name, "n": len(g), "fake_share": g.label.mean(), "eer": eer(g.label, g.score)[0]})
spk = pd.DataFrame(spk).sort_values("eer")
spk.to_csv(ROOT / "results/e5_itw_speakers.csv", index=False, float_format="%.4f")
print("speakers with EER < 20%:", (spk.eer < 0.2).sum(), "| above 50% (worse than chance):", (spk.eer > 0.5).sum(), "of", len(spk))
pd.concat([spk.head(5), spk.tail(5)]).round(3)

In [ ]:
# the most confident mistakes of rung 2 on eval, at its dev threshold
thr2 = eer(EV["dev"].label, s2["dev"])[1]
ev = EV["eval"][["file", "label", "attack", "speaker"]].assign(score=s2["eval"], duration=silf["eval"].duration,
                                                                 lead_sil=silf["eval"].lead_sil)
fa = ev[(ev.label == 1) & (ev.score < thr2)].sort_values("score").head(200)   # fakes let through
fr = ev[(ev.label == 0) & (ev.score >= thr2)].sort_values("score", ascending=False).head(200)  # real flagged
print(f"dev threshold {thr2:.3f}: {len(ev[(ev.label == 1) & (ev.score < thr2)])} fakes accepted, {len(ev[(ev.label == 0) & (ev.score >= thr2)])} real clips rejected")
print("most confident false accepts, by attack:", fa.attack.value_counts().head(6).to_dict())
print("median duration / lead silence: false accepts", fa.duration.median().round(2), fa.lead_sil.median().round(2),
      "| false rejects", fr.duration.median().round(2), fr.lead_sil.median().round(2),
      "| all eval", ev.duration.median().round(2), ev.lead_sil.median().round(2))
print("false rejects: speakers", fr.speaker.nunique(), "| top:", fr.speaker.value_counts().head(5).to_dict())

## Figures

In [ ]:
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e1e0d9"
plt.rcParams.update({"font.size": 9, "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK,
                     "xtick.color": MUTED, "ytick.color": MUTED, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.facecolor": "white"})
e5 = pd.read_csv(ROOT / "results/e5.csv")
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6), gridspec_kw={"width_ratios": [1.4, 1]})
conds = ["eval", "eval_trim", "eval_gain-12", "eval_gain+6", "itw", "itw_trim"]
labels = ["eval", "eval\ntrimmed", "eval\n-12 dB", "eval\n+6 dB", "ITW", "ITW\ntrimmed"]
series = [("hgb:lfcc40dd_e3devsel", "rung 1 (LFCC)", "#eda100", None),
          ("hgb:lfcc40dd_e3devsel_augment", "rung 1 + augmentation", "#eda100", "////"),
          ("logreg:xlsr300m_L03", "rung 2 (XLS-R)", "#1baf7a", None)]
w = 0.27
for j, (mname, lab, col, hatch) in enumerate(series):
    d = e5[e5.model == mname].set_index("split")
    x = np.arange(len(conds)) + (j - 1) * w
    vals = [100 * d.eer.get(c, np.nan) for c in conds]
    ax[0].bar(x, vals, width=w - 0.02, color=col, hatch=hatch, edgecolor="white", label=lab)
    for xi, v in zip(x, vals):
        if not np.isnan(v):
            ax[0].text(xi, v + 1, f"{v:.0f}", ha="center", fontsize=7, color=INK)
ax[0].axhline(50, color=MUTED, ls=":", lw=1)
ax[0].set_xticks(range(len(conds)), labels); ax[0].set_ylim(0, 95); ax[0].set_ylabel("EER %")
ax[0].grid(axis="y", color=GRID, linewidth=0.6); ax[0].legend(frameon=False, fontsize=8, loc="upper left")
ax[0].set_title("(a) Test-time changes and augmentation (XLS-R is gain-invariant: no gain bars)", loc="left", color=INK)

for s, col, lab in [("dev", "#c3c2b7", "ASVspoof dev"), ("eval", "#52514e", "ASVspoof eval"), ("itw", "#e34948", "In-the-Wild")]:
    r = reliability(EV[s].label, PROBS[2][s])
    e = cal[(cal.rung == 2) & (cal.split == s)].ece_before.iloc[0]
    ax[1].plot(r.mean_prob, r.frac_fake, marker="o", ms=4, lw=1.8, color=col, label=f"{lab} (ECE {100 * e:.0f}%)")
ax[1].plot([0, 1], [0, 1], color=MUTED, ls="--", lw=1)
ax[1].set_xlabel("predicted P(fake)"); ax[1].set_ylabel("observed fake rate")
ax[1].grid(color=GRID, linewidth=0.6); ax[1].legend(frameon=False, fontsize=8, loc="lower right")
ax[1].set_title("(b) Reliability of rung 2 (before scaling)", loc="left", color=INK)
fig.tight_layout()
fig.savefig(ROOT / "results/figures/e5_shortcuts_calibration.png", dpi=200)
fig.savefig(ROOT / "results/figures/e5_shortcuts_calibration.pdf")